# Giai đoạn 1: Trích xuất Dữ liệu từ Mạng (PhysioNet)
Đây là Notebook mô tả quá trình lấy dữ liệu Điện tim (ECG) từ bộ cơ sở dữ liệu chuẩn quốc tế **MIT-BIH Atrial Fibrillation Database (afdb)**.

### Mục tiêu:
- Tải trực tiếp tín hiệu (Streaming) từ Đại học MIT.
- Thực hiện Tiền xử lý (Lọc nhiễu sinh lý, chia cửa sổ 30 giây).
- Trích xuất 10 đặc trưng HRV bằng toán học (Numpy/Scipy).
- Lưu lại thành file CSV để Nhóm tự Huấn luyện ở Giai đoạn 2.

In [ ]:
import os
import wfdb
import numpy as np
import pandas as pd
from scipy import signal
from scipy.interpolate import interp1d

# Tạo thư mục chứa dữ liệu
os.makedirs('../../data/features', exist_ok=True)
os.makedirs('../../data/external/afdb', exist_ok=True)

### 1. Hàm Trích xuất Đặc trưng (Feature Engineering)
Hàm này được viết bằng toán học thuần túy (không dùng thư viện ngoài) để đảm bảo có thể chuyển sang ngôn ngữ C++ chạy trên vi điều khiển ESP32 sau này.

In [ ]:
def extract_hrv_features(valid_rr):
    # 1. Time-domain (Miền thời gian)
    rr_diff = np.diff(valid_rr)
    mean_nni = np.mean(valid_rr)
    sdnn = np.std(valid_rr, ddof=1)
    rmssd = np.sqrt(np.mean(rr_diff**2))
    pnn50 = (np.sum(np.abs(rr_diff) > 50) / len(rr_diff)) * 100
    cvnni = sdnn / mean_nni if mean_nni > 0 else 0
    cvsd = rmssd / mean_nni if mean_nni > 0 else 0
    
    # 2. Frequency-domain (Miền tần số bằng phương pháp Welch)
    try:
        time_arr = np.cumsum(valid_rr) / 1000.0
        fs_interp = 4.0
        time_interp = np.arange(time_arr[0], time_arr[-1], 1/fs_interp)
        
        if len(time_interp) < 10:
            return None
            
        f_interp = interp1d(time_arr, valid_rr, kind='cubic', fill_value="extrapolate")
        rr_interp = f_interp(time_interp)
        
        nperseg = min(len(rr_interp), 256)
        f, pxx = signal.welch(rr_interp, fs=fs_interp, nperseg=nperseg)
        
        lf_band = (f >= 0.04) & (f < 0.15)
        hf_band = (f >= 0.15) & (f < 0.4)
        
        lf = np.trapz(pxx[lf_band], f[lf_band]) if np.sum(lf_band) > 0 else 0
        hf = np.trapz(pxx[hf_band], f[hf_band]) if np.sum(hf_band) > 0 else 0
        lf_hf_ratio = lf / hf if hf > 0 else 0
        total_power = np.trapz(pxx, f)
        
    except Exception:
        lf = hf = lf_hf_ratio = total_power = 0
        
    return {
        'mean_nni': mean_nni, 'sdnn': sdnn, 'rmssd': rmssd, 'pnn50': pnn50,
        'cvnni': cvnni, 'cvsd': cvsd, 'lf': lf, 'hf': hf,
        'lf_hf_ratio': lf_hf_ratio, 'total_power': total_power
    }

### 2. Tiền xử lý (Preprocessing)
Đọc nhãn bệnh lý, chia cửa sổ 30 giây, lọc nhiễu sinh lý (<300ms và >2000ms).

In [ ]:
WINDOW_SIZE_SEC = 30

def process_record(record_name):
    print(f"Đang xử lý bản ghi: {record_name}")
    try:
        qrs_ann = wfdb.rdann(record_name, 'qrs', pn_dir='afdb')
        atr_ann = wfdb.rdann(record_name, 'atr', pn_dir='afdb')
        fs = qrs_ann.fs
        r_peaks = qrs_ann.sample
        rhythm_samples = atr_ann.sample
        rhythm_labels = atr_ann.aux_note
        
        features_list = []
        total_samples = max(r_peaks) if len(r_peaks) > 0 else 0
        window_size_samples = WINDOW_SIZE_SEC * fs
        
        for start_idx in range(0, total_samples, window_size_samples):
            end_idx = start_idx + window_size_samples
            valid_labels = [label for i, label in enumerate(rhythm_labels) if rhythm_samples[i] <= end_idx]
            
            if not valid_labels:
                continue
            
            current_rhythm = valid_labels[-1]
            if current_rhythm not in ['(AFIB', '(N']:
                continue
                
            label = 'AFib' if current_rhythm == '(AFIB' else 'Normal'
            window_peaks = r_peaks[(r_peaks >= start_idx) & (r_peaks < end_idx)]
            
            if len(window_peaks) < 10: 
                continue
                
            rr_intervals_ms = (np.diff(window_peaks) / fs) * 1000
            
            # BỘ LỌC SINH LÝ HỌC (Physiological Filter)
            valid_rr = rr_intervals_ms[(rr_intervals_ms >= 300) & (rr_intervals_ms <= 2000)]
            
            if len(valid_rr) < 10:
                continue
            
            features = extract_hrv_features(valid_rr)
            if features is not None:
                features['label'] = label
                features['record'] = record_name
                features_list.append(features)
                
        return features_list
    except Exception as e:
        print(f"Lỗi: {e}")
        return []

### 3. Thực thi kéo dữ liệu từ MIT-BIH

In [ ]:
# Lấy danh sách từ PhysioNet
records = wfdb.get_record_list('afdb')
print(f"Đã tìm thấy {len(records)} bản ghi.")

all_features = []
for record in records: # Chạy qua toàn bộ 25 bệnh nhân
    features = process_record(record)
    all_features.extend(features)

df = pd.DataFrame(all_features)
print(f"\nTổng số mẫu thu được: {len(df)}")

if len(df) > 0:
    output_path = '../../data/features/mitbih_af_features.csv'
    df.to_csv(output_path, index=False)
    print(f"Đã lưu thành công vào: {output_path}")